In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# TotalCharges kabhi kabhi text ke roop mein aata hai, usko number bana do
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# customerID hata do, kaam ka nahi
df = df.drop('customerID', axis=1)

# Target column (jo predict karna hai)
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop('Churn', axis=1)

# Numerical aur categorical columns alag karo
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_cols = [col for col in X.columns if col not in numerical_cols]

print(numerical_cols)
print(categorical_cols)

['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:
# Preprocessing: numerical ko scale karo, categorical ko one-hot encode karo
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# Pipeline: preprocessing + model ek object mein
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

print("Pipeline ready hai")

Pipeline ready hai


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Pipeline Accuracy: {accuracy:.2%}")

Pipeline Accuracy: 82.19%


In [7]:
# Feature 1: Average monthly spend (TotalCharges / tenure)
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)  # +1 taake naye customer (tenure=0) division error na de

# Feature 2: Naya customer hai ya nahi (tenure < 6 mahine)
df['IsNewCustomer'] = (df['tenure'] < 6).astype(int)

# Ab dobara X aur y banao in naye features ke sath
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop('Churn', axis=1)

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'AvgMonthlySpend', 'IsNewCustomer']
categorical_cols = [col for col in X.columns if col not in numerical_cols]

print("Naye features add ho gaye")
print(df[['AvgMonthlySpend', 'IsNewCustomer']].head())

Naye features add ho gaye
   AvgMonthlySpend  IsNewCustomer
0        14.925000              1
1        53.985714              0
2        36.050000              1
3        40.016304              0
4        50.550000              1


In [8]:
# Naya pipeline (updated numerical_cols ke sath)
preprocessor2 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

pipeline2 = Pipeline(steps=[
    ('preprocessing', preprocessor2),
    ('model', LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline2.fit(X_train, y_train)

y_pred2 = pipeline2.predict(X_test)
accuracy2 = accuracy_score(y_test, y_pred2)
print(f"Naye Features ke Sath Accuracy: {accuracy2:.2%}")
print(f"Purani Accuracy (bina naye features): 82.19%")

Naye Features ke Sath Accuracy: 81.48%
Purani Accuracy (bina naye features): 82.19%


In [9]:
import joblib

joblib.dump(pipeline, 'churn_pipeline.pkl')
print("Pipeline save ho gaya: churn_pipeline.pkl")

from google.colab import files
files.download('churn_pipeline.pkl')

Pipeline save ho gaya: churn_pipeline.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>